# RNNs Explained — Building One From Scratch vs. Using Pretrained Embeddings

**Audience:** this notebook is a teaching aid for candidates — it walks through *what a
Recurrent Neural Network actually is*, *why it's useful*, and then builds **two working
models side by side** on the same task so you can see the difference for yourself:

1. **From scratch** — the model learns everything, including what words even mean
   (its embeddings), purely from our small labeled dataset.
2. **Pretrained** — the model starts from word embeddings (GloVe) that were already
   trained on billions of words of real text, and only has to learn the task on top of
   that head start.

**The task we'll use to make this concrete:** intent classification for AI telecaller
conversations — the same domain as `ai_telecaller_poc.ipynb` in this repo. Given a short
sentence a caller says, predict which of 7 intents it is:

`fee_question`, `schedule_question`, `certification_question`, `opt_out`,
`interested`, `not_interested`, `general_offtopic`

This is a real, useful classifier — you could drop this straight into the `generate_response`
step of the POC notebook to route caller utterances before they even hit the LLM (e.g.
instantly flag `opt_out` without waiting on an LLM call at all).

Runs fine on CPU — no GPU required. Everything here is intentionally small and readable
over "production-grade," because the goal is to build correct intuition, not to ship a
polished model.

## 1. What is an RNN, and why do we need it?

A normal feedforward network (or a plain classifier over a bag of words) looks at its
input all at once and has no notion of *order* or *memory*. But language is a sequence —
"not interested" and "interested, not... actually no, not interested" mean very
different things, and word **order** and **context carried forward** matter.

A **Recurrent Neural Network** processes a sequence one element (e.g. one word) at a
time, and carries a **hidden state** `h` forward from each step to the next — a running
summary of everything seen so far. That's the "memory."

At every timestep `t`, given the current input `x_t` and the previous hidden state
`h_{t-1}`, an RNN computes:

```
h_t = tanh( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )
y_t = W_hy · h_t  +  b_y            (only needed at steps where you want an output)
```

- `W_xh` — how much the *current input* matters
- `W_hh` — how much the *previous memory* matters (this is what makes it "recurrent" —
  the same weight matrix is reused at every timestep)
- `h_t` — the new hidden state / memory, passed on to the next timestep

**"Unrolled" in time**, for a 4-word sentence, it looks like this — the same cell,
reused at every step, connected by the hidden state:

```
x1        x2        x3        x4
 |         |         |         |
 v         v         v         v
[RNN]-h1->[RNN]-h2->[RNN]-h3->[RNN]-h4->  (h4 = summary of the whole sentence)
```

For a **classification** task like ours, we only care about the *final* hidden state
`h4` — by the last word, it has (hopefully) absorbed enough of the sentence to tell us
the intent. That final hidden state is what we feed into a small classifier head.

**Training** an RNN uses **backpropagation through time (BPTT)**: unroll the recurrence
for the whole sequence, then backpropagate the loss through every timestep, back to
`t=1`. Because the *same* weights (`W_xh`, `W_hh`) are reused at every step, their
gradients from every timestep get summed together.

**The classic weakness — vanishing/exploding gradients:** with long sequences, that
repeated multiplication by `W_hh` at every backward step can shrink gradients toward
zero (or blow them up), which is why plain ("vanilla") RNNs struggle to remember
anything from far back in a long sequence. This is exactly why **LSTM** and **GRU**
cells were invented (they add gates that let gradients flow more directly), and why
**Transformers** have since replaced RNNs in most state-of-the-art NLP systems.
We'll still build a vanilla RNN here because the vanilla recurrence is the clearest
place to *see* the core idea — everything else (LSTM, GRU, even attention) is a
variation on "carry a state forward and update it at every step."

**Where RNNs (and their descendants) show up in practice:**
- Text: sentiment/intent classification, language modeling, translation
- Speech: acoustic modeling, the recurrent layers inside older speech-to-text systems
- Time series: forecasting, anomaly detection on sequences of sensor/metric readings
- Anywhere your input is naturally an ordered sequence and context accumulates over time

Let's build one.

In [ ]:
# --- Setup ---
import random
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. See the recurrence equation as actual code, before we use PyTorch's version

Before reaching for `nn.RNN`, let's hand-write the exact same math in plain NumPy on
tiny numbers, so there's no black box. Then we'll copy those same weights into
PyTorch's built-in `nn.RNNCell` and confirm they produce *identical* outputs — proving
`nn.RNN` isn't doing anything magical, just this same loop, fast and vectorized.

In [ ]:
# Tiny toy dimensions, just for this demonstration
input_dim = 3   # size of each x_t (e.g. a tiny word embedding)
hidden_dim = 4  # size of the hidden state h_t
seq_len = 5     # a 5-token toy "sentence"

rng = np.random.default_rng(0)
W_xh = rng.normal(scale=0.5, size=(hidden_dim, input_dim))
W_hh = rng.normal(scale=0.5, size=(hidden_dim, hidden_dim))
b_h = np.zeros(hidden_dim)

x_sequence = rng.normal(size=(seq_len, input_dim))  # 5 toy input vectors

def rnn_cell_step(x_t, h_prev):
    '''One RNN timestep, written out exactly as in the equation above.'''
    return np.tanh(W_xh @ x_t + W_hh @ h_prev + b_h)

h = np.zeros(hidden_dim)  # h_0 starts at all zeros
print("Rolling the RNN forward one timestep at a time:\n")
for t in range(seq_len):
    h = rnn_cell_step(x_sequence[t], h)
    print(f"  t={t}  x_t={np.round(x_sequence[t], 2)}  ->  h_t={np.round(h, 3)}")

h_final_numpy = h.copy()
print(f"\nFinal hidden state after all {seq_len} steps (this is the sentence summary): "
      f"{np.round(h_final_numpy, 3)}")

In [ ]:
# --- Now do the exact same computation with PyTorch's built-in RNNCell, ---
# --- loading in the SAME weights, to prove it's the same math. ---
torch_cell = nn.RNNCell(input_dim, hidden_dim, nonlinearity="tanh")
with torch.no_grad():
    torch_cell.weight_ih.copy_(torch.tensor(W_xh, dtype=torch.float32))
    torch_cell.weight_hh.copy_(torch.tensor(W_hh, dtype=torch.float32))
    torch_cell.bias_ih.zero_()
    torch_cell.bias_hh.zero_()

h_t = torch.zeros(1, hidden_dim)
x_t_all = torch.tensor(x_sequence, dtype=torch.float32)
for t in range(seq_len):
    h_t = torch_cell(x_t_all[t].unsqueeze(0), h_t)

h_final_torch = h_t.squeeze(0).detach().numpy()
print("Final hidden state (NumPy, hand-written): ", np.round(h_final_numpy, 4))
print("Final hidden state (PyTorch nn.RNNCell):  ", np.round(h_final_torch, 4))

assert np.allclose(h_final_numpy, h_final_torch, atol=1e-5), \
    "Mismatch — PyTorch's RNNCell should reproduce the exact same recurrence."
print("\nMATCH confirmed: nn.RNNCell is exactly the loop above, just optimized.")

## 3. The dataset — caller intents

A small, hand-labeled dataset of caller-style sentences across the 7 intents. In a real
deployment you'd pull these from actual call transcripts (see `ai_telecaller_poc.ipynb`
and `telephony_bridge/`) — here they're written by hand so the notebook is fully
self-contained and reproducible for teaching.

This is intentionally **small** (12 examples per class). That's on purpose: it's exactly
the situation where the "pretrained vs. from scratch" comparison in Section 6 becomes
interesting — with only a handful of labeled examples per class, a from-scratch model
barely has enough data to learn what words even mean, let alone the task.

In [ ]:
INTENT_EXAMPLES = {
    "fee_question": [
        "How much does this course cost?",
        "What's the fee for the programme?",
        "Is there any discount on the price?",
        "How much do I need to pay?",
        "What is the total cost including taxes?",
        "Can you tell me the fees please?",
        "Is the course expensive?",
        "What's the price for the bootcamp?",
        "How much would this training cost me?",
        "Do you have EMI options for the fee?",
        "What is the enrollment fee?",
        "Can I know the exact amount to pay?",
    ],
    "schedule_question": [
        "When does the batch start?",
        "What are the class timings?",
        "How long is the course?",
        "When will the programme begin?",
        "What days do the classes happen?",
        "Is it a weekday or weekend batch?",
        "How many weeks does the training run?",
        "What time do the sessions start?",
        "When is the next batch starting?",
        "How many hours per week is the commitment?",
        "Is there a morning batch available?",
        "What's the duration of the course?",
    ],
    "certification_question": [
        "Will I get a certificate after finishing?",
        "Is there a certification at the end?",
        "Do you provide a completion certificate?",
        "Is the certificate recognized by companies?",
        "What credential do I receive after the course?",
        "Is there an exam before getting certified?",
        "Does the certificate help with job applications?",
        "Who issues the certificate?",
        "Is the certification globally valid?",
        "Do I need to pass a test to get certified?",
        "What does the certificate look like?",
        "Is certification included in the fee?",
    ],
    "opt_out": [
        "Please remove my number from your list.",
        "Stop calling me, I'm not interested.",
        "Take me off your calling list.",
        "Don't call me again.",
        "I want to opt out of these calls.",
        "Please do not contact me anymore.",
        "Remove me from your database.",
        "Stop contacting me about this course.",
        "I did not consent to these calls, remove my number.",
        "Unsubscribe me from your calls.",
        "Do not call this number again.",
        "I want my number deleted from your records.",
    ],
    "interested": [
        "Yes, I'm interested, tell me more.",
        "This sounds great, how do I enroll?",
        "I would like to sign up for this programme.",
        "Sounds good, send me the details.",
        "I'm quite interested, what's the next step?",
        "Yes please, I want to join.",
        "I'd love to learn more about this.",
        "Count me in, how do I register?",
        "Yes, this is exactly what I was looking for.",
        "I want to proceed with enrollment.",
        "This is helpful, I'm keen to join.",
        "Yes, please share the registration link.",
    ],
    "not_interested": [
        "I'm not interested right now, thanks.",
        "This isn't for me at the moment.",
        "No thanks, I don't need this course.",
        "I already have a job, not looking for training.",
        "Not interested, but thanks for calling.",
        "I don't think this is the right fit for me.",
        "No, I'll pass on this one.",
        "Maybe some other time, not now.",
        "I'm not looking to enroll currently.",
        "Not really interested in this programme.",
        "I'll skip this for now, thank you.",
        "This isn't something I need right now.",
    ],
    "general_offtopic": [
        "What's the capital of Australia?",
        "Can you tell me a joke?",
        "What's the weather like today?",
        "Who won the cricket match yesterday?",
        "What time is it right now?",
        "Do you know any good restaurants nearby?",
        "What's your favorite movie?",
        "How's the traffic in the city today?",
        "Can you recommend a good book?",
        "What is the population of India?",
        "Tell me something interesting.",
        "Is it going to rain tomorrow?",
    ],
}

LABELS = sorted(INTENT_EXAMPLES.keys())
label_to_idx = {label: i for i, label in enumerate(LABELS)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

all_examples = [(text, label) for label, texts in INTENT_EXAMPLES.items() for text in texts]
random.shuffle(all_examples)

split = int(0.8 * len(all_examples))
train_examples = all_examples[:split]
test_examples = all_examples[split:]

print(f"Total examples: {len(all_examples)}  |  train: {len(train_examples)}  |  test: {len(test_examples)}")
print(f"Labels ({len(LABELS)}): {LABELS}")

## 4. Preprocessing — turning text into padded integer sequences

Steps, same as any text pipeline:
1. Lowercase + tokenize on whitespace/punctuation.
2. Build a vocabulary **from the training set only** (never let test-set words leak in —
   in a real system those are words you haven't seen yet).
3. Map each token to an integer id (`0` reserved for padding, `1` for unknown words).
4. Pad each batch's sequences to the same length so they can be stacked into a tensor,
   and pack them so the RNN knows each sequence's *real* length (so padding doesn't
   pollute the final hidden state).

In [ ]:
import re

def tokenize(text: str):
    return re.findall(r"[a-z']+", text.lower())

PAD_IDX, UNK_IDX = 0, 1

def build_vocab(examples):
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for text, _ in examples:
        for tok in tokenize(text):
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab

vocab = build_vocab(train_examples)
print(f"Vocab size (from training data only): {len(vocab)}")

def encode(text: str):
    return [vocab.get(tok, UNK_IDX) for tok in tokenize(text)]

class IntentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        text, label = self.examples[i]
        ids = encode(text)
        return torch.tensor(ids, dtype=torch.long), label_to_idx[label]

def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

train_loader = DataLoader(IntentDataset(train_examples), batch_size=8, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(IntentDataset(test_examples), batch_size=8, shuffle=False, collate_fn=collate_batch)

# Sanity check on one batch
sample_padded, sample_lengths, sample_labels = next(iter(train_loader))
print("Sample padded batch shape:", sample_padded.shape)
print("Sample lengths:", sample_lengths.tolist())
print("Sample labels:", [idx_to_label[i.item()] for i in sample_labels])

## 5. Model A — RNN trained fully from scratch

Architecture — the same shape we'll reuse for Model B, so the *only* difference between
the two models is where the embedding weights start out:

```
token ids -> Embedding (learned) -> RNN -> final hidden state -> Linear -> 7 intent scores
```

"From scratch" here means the `Embedding` layer starts at **random** vectors and has to
learn what every word means — purely from the ~65 training sentences we have. There's no
outside knowledge at all; if a word never appears in training, its embedding never moves
from its random initialization.

In [ ]:
EMBED_DIM = 32
HIDDEN_DIM = 32
NUM_CLASSES = len(LABELS)

class IntentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pretrained_embeddings=None, freeze_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
            self.embedding.weight.requires_grad = not freeze_embeddings
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, padded_ids, lengths):
        embedded = self.embedding(padded_ids)  # (batch, seq_len, embed_dim)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.rnn(packed)  # h_n: (1, batch, hidden_dim) — final hidden state per sequence
        final_hidden = h_n.squeeze(0)
        return self.classifier(final_hidden)

def train_model(model, train_loader, test_loader, epochs=40, lr=0.01, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for padded, lengths, labels in train_loader:
            padded, lengths, labels = padded.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(padded, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        test_acc = evaluate_model(model, test_loader)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == epochs):
            print(f"  epoch {epoch:3d}/{epochs}  train_loss={train_loss:.3f}  "
                  f"train_acc={train_acc:.2f}  test_acc={test_acc:.2f}")

    return history

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for padded, lengths, labels in loader:
            padded, lengths, labels = padded.to(device), lengths.to(device), labels.to(device)
            logits = model(padded, lengths)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total

print("Model A architecture:")
model_a = IntentRNN(len(vocab), EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model_a)

In [ ]:
print("Training Model A (from scratch, random embeddings)...\n")
history_a = train_model(model_a, train_loader, test_loader, epochs=40, lr=0.01)
final_test_acc_a = history_a["test_acc"][-1]
print(f"\nModel A final test accuracy: {final_test_acc_a:.2%}")

## 6. Model B — the same RNN, but starting from pretrained GloVe embeddings

This is **transfer learning**: instead of random-initializing the embedding layer, we
initialize it with **GloVe** vectors — word embeddings pretrained by Stanford on 6
billion tokens of Wikipedia + Gigaword text. Those vectors already encode a lot of
general semantic structure (similar words end up with similar vectors) before our model
has seen a single one of our own training sentences.

Everything else about the architecture is **identical** to Model A — same `IntentRNN`
class, same hidden size, same training loop. The *only* difference is the starting point
of the embedding weights. That controlled comparison is the point: any difference we see
in Section 7 is attributable to the pretrained initialization, not to a different model.

We fine-tune the embeddings here (`freeze_embeddings=False`) rather than freezing them —
letting them adjust slightly to our task usually works a little better than freezing
them outright, though freezing is worth trying yourself (see the exercises at the end).

**Needs internet access** to download GloVe the first time (~66MB for the 50-dim
vectors). If that's not available in your environment, the `try/except` below falls back
to random vectors with a clear warning — the rest of the notebook still runs, but the
comparison in Section 7 won't show the real effect.

In [ ]:
# One-time install if gensim isn't already available (uncomment in Colab):
# !pip -q install gensim

GLOVE_DIM = 50
glove_vectors = None

try:
    import gensim.downloader as gensim_api
    print("Downloading/loading GloVe (glove-wiki-gigaword-50)... this can take a minute the first time.")
    glove_vectors = gensim_api.load("glove-wiki-gigaword-50")
    print(f"Loaded GloVe: {len(glove_vectors)} words, {GLOVE_DIM} dimensions.")
except Exception as e:
    print(f"Could not load GloVe ({e!r}).")
    print("Falling back to random embeddings for Model B — the pretrained-vs-scratch")
    print("comparison below will no longer be meaningful. Run this in an environment")
    print("with internet access to see the real transfer-learning effect.")

def build_pretrained_embedding_matrix(vocab, glove, embed_dim, verbose=True):
    matrix = torch.empty(len(vocab), embed_dim).normal_(mean=0.0, std=0.1)
    matrix[PAD_IDX].zero_()
    found = 0
    for word, idx in vocab.items():
        if glove is not None and word in glove:
            matrix[idx] = torch.tensor(glove[word].copy())
            found += 1
    coverage = found / len(vocab) if len(vocab) else 0.0
    if verbose:
        print(f"Pretrained coverage: {found}/{len(vocab)} vocab words found in GloVe ({coverage:.1%}).")
    return matrix

pretrained_matrix = build_pretrained_embedding_matrix(vocab, glove_vectors, GLOVE_DIM)

In [ ]:
print("Model B architecture (same as Model A, different embedding init):")
model_b = IntentRNN(
    len(vocab), GLOVE_DIM, HIDDEN_DIM, NUM_CLASSES,
    pretrained_embeddings=pretrained_matrix, freeze_embeddings=False,
)
print(model_b)

print("\nTraining Model B (pretrained GloVe embeddings, fine-tuned)...\n")
history_b = train_model(model_b, train_loader, test_loader, epochs=40, lr=0.01)
final_test_acc_b = history_b["test_acc"][-1]
print(f"\nModel B final test accuracy: {final_test_acc_b:.2%}")

## 7. Compare the two side by side

One caveat before looking at this: with only ~80 sentences total, a *single* train/test
split is small enough to be noisy — which exact sentences land in the 20% test slice can
swing accuracy by a lot. Section 8 below repeats this comparison across several
independent splits to give a more trustworthy answer; treat the single-run numbers here
as illustrative of the training dynamics, not as the final verdict.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_a["train_loss"], label="Model A (from scratch)")
axes[0].plot(history_b["train_loss"], label="Model B (pretrained GloVe)")
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].legend()

axes[1].plot(history_a["test_acc"], label="Model A (from scratch)")
axes[1].plot(history_b["test_acc"], label="Model B (pretrained GloVe)")
axes[1].set_title("Held-out test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final test accuracy — Model A (from scratch):     {final_test_acc_a:.2%}")
print(f"Final test accuracy — Model B (pretrained GloVe):  {final_test_acc_b:.2%}")

**What to look for in the plots above** (results vary run to run with a dataset
this small, but the pattern usually holds):

- **Model B (pretrained) tends to reach lower loss / higher accuracy faster.** It isn't
  spending early epochs figuring out that "cost" and "price" or "fee" and "expensive"
  are related — GloVe already put those words near each other in vector space. It only
  has to learn the *mapping from meaning to intent label*, not word meaning itself.
- **Model A (from scratch) has strictly less to work with**: ~65 training sentences is
  nowhere near enough to learn good word representations *and* the task at the same
  time. It can still fit the training set, but often generalizes worse to the held-out
  test sentences (watch for the test-accuracy curve being noisier or lower).
- **This gap is exactly why transfer learning matters in practice.** Labeled data is the
  expensive, slow part of any ML project (in this repo's context: transcribing and
  hand-labeling real caller calls). Pretrained embeddings — and today, pretrained large
  language models more generally — let you get good task performance with far less
  labeled data than training a model from zero.

**When would you still train from scratch?** When your domain's vocabulary is so
different from general text that pretrained embeddings don't transfer well (e.g. heavy
code-mixing, a low-resource language with no good pretrained vectors, or highly
specialized jargon), or when you have *abundant* labeled data and want a model with no
outside dependencies. In this project's case — a real deployment would likely have
Telugu/English code-mixed transcripts, where you'd want to check whether a suitable
pretrained embedding even exists before assuming Model B's advantage holds.

## 8. Don't trust a single run — repeat over multiple random splits

The single comparison above is a good way to *see* the training dynamics (the loss
curves, how quickly each model fits), but with a dataset this small it's genuinely easy
for one lucky or unlucky test split to flip which model "wins." That's not a flaw in the
models — it's a property of evaluating on ~17 examples. The fix practitioners actually
use is simple: repeat the whole experiment (new split, new model, fresh training) across
several random seeds, and compare the **average**.

Below we do exactly that: a fresh **stratified** split (same proportion of every intent
in train and test, every time) for each of `N_SEEDS` seeds, a fresh vocabulary, a fresh
pretrained-embedding matrix, and fresh training for both approaches — then we compare
mean test accuracy (± standard deviation) across all of them. This takes a little longer
to run than Section 7, but the result is far more trustworthy than any single number
above. Watch the per-seed printout too: it's normal for the "worse" approach to win on
an individual seed here and there — what matters is the average.

In [ ]:
def stratified_split(seed, test_frac=0.2):
    '''Same idea as the train/test split in Section 3, but guarantees every intent is
    represented in the same proportion in both train and test, for every seed.'''
    rnd = random.Random(seed)
    train_ex, test_ex = [], []
    for label, texts in INTENT_EXAMPLES.items():
        shuffled = texts[:]
        rnd.shuffle(shuffled)
        n_test = max(1, round(test_frac * len(shuffled)))
        test_ex += [(t, label) for t in shuffled[:n_test]]
        train_ex += [(t, label) for t in shuffled[n_test:]]
    rnd.shuffle(train_ex)
    rnd.shuffle(test_ex)
    return train_ex, test_ex

class SplitDataset(Dataset):
    '''Like IntentDataset, but takes its own vocab instead of relying on the module-level
    `vocab` — each seed below builds a fresh vocab from its own train split.'''
    def __init__(self, examples, seed_vocab):
        self.examples = examples
        self.seed_vocab = seed_vocab

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        text, label = self.examples[i]
        ids = [self.seed_vocab.get(tok, UNK_IDX) for tok in tokenize(text)]
        return torch.tensor(ids, dtype=torch.long), label_to_idx[label]

N_SEEDS = 10
seed_results_a, seed_results_b = [], []

print(f"Repeating the from-scratch vs. pretrained comparison over {N_SEEDS} independent "
      f"random splits...\n")

for seed in range(N_SEEDS):
    random.seed(seed)
    np.random.seed(seed)

    seed_train, seed_test = stratified_split(seed)
    seed_vocab = build_vocab(seed_train)
    seed_train_loader = DataLoader(SplitDataset(seed_train, seed_vocab), batch_size=8,
                                    shuffle=True, collate_fn=collate_batch)
    seed_test_loader = DataLoader(SplitDataset(seed_test, seed_vocab), batch_size=8,
                                   shuffle=False, collate_fn=collate_batch)

    torch.manual_seed(seed)
    scratch_model = IntentRNN(len(seed_vocab), EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
    train_model(scratch_model, seed_train_loader, seed_test_loader, epochs=40, lr=0.01, verbose=False)
    acc_a = evaluate_model(scratch_model, seed_test_loader)

    seed_pretrained_matrix = build_pretrained_embedding_matrix(
        seed_vocab, glove_vectors, GLOVE_DIM, verbose=False
    )
    torch.manual_seed(seed)
    pretrained_model = IntentRNN(len(seed_vocab), GLOVE_DIM, HIDDEN_DIM, NUM_CLASSES,
                                  pretrained_embeddings=seed_pretrained_matrix, freeze_embeddings=False)
    train_model(pretrained_model, seed_train_loader, seed_test_loader, epochs=40, lr=0.01, verbose=False)
    acc_b = evaluate_model(pretrained_model, seed_test_loader)

    seed_results_a.append(acc_a)
    seed_results_b.append(acc_b)
    print(f"  seed {seed}: from-scratch={acc_a:.2f}   pretrained={acc_b:.2f}"
          f"{'   <- pretrained lower this time' if acc_b < acc_a else ''}")

mean_a, std_a = float(np.mean(seed_results_a)), float(np.std(seed_results_a))
mean_b, std_b = float(np.mean(seed_results_b)), float(np.std(seed_results_b))
wins_b = sum(1 for a, b in zip(seed_results_a, seed_results_b) if b > a)

print(f"\nFrom scratch — mean test accuracy: {mean_a:.2%}  (std {std_a:.2%}) over {N_SEEDS} seeds")
print(f"Pretrained   — mean test accuracy: {mean_b:.2%}  (std {std_b:.2%}) over {N_SEEDS} seeds")
print(f"Pretrained beat from-scratch on {wins_b}/{N_SEEDS} individual seeds")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
bars = ax.bar(
    ["From scratch", "Pretrained\n(GloVe)"],
    [mean_a, mean_b],
    yerr=[std_a, std_b],
    capsize=8,
    color=["#888888", "#4C72B0"],
)
ax.set_ylabel("test accuracy (mean ± std over seeds)")
ax.set_ylim(0, 1)
ax.set_title(f"Averaged over {N_SEEDS} independent train/test splits")
for bar, mean in zip(bars, [mean_a, mean_b]):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.03, f"{mean:.1%}", ha="center")
plt.tight_layout()
plt.show()

This is the number worth trusting more than the single-run plot in Section 7: on
average, the pretrained-embedding model tends to generalize a bit better than the
from-scratch one on this small dataset — but not on *every* seed, and the gap is modest,
not dramatic. That combination (real average advantage, high per-run variance, small
absolute gap) is exactly what you should expect from a dataset this tiny, and it's a
useful, honest thing to be able to say about any small-data ML result you're evaluating:
"what's the average over multiple runs, and how much does it vary?" is a more meaningful
question than "what did I get on my one run?"

If you increase the dataset size (see the exercises in Section 10), you should see both
the per-seed variance shrink and the pretrained model's advantage become more consistent
— worth testing to confirm that intuition yourself.

## 9. Try it yourself — live predictions from both models

Type a caller-style sentence and see what each model predicts, with its confidence.
Try something ambiguous (e.g. "how much time and how much money") and see where the two
models agree or disagree.

In [ ]:
def predict_intent(model, text: str):
    model.eval()
    ids = torch.tensor([encode(text)], dtype=torch.long)
    lengths = torch.tensor([ids.shape[1]], dtype=torch.long)
    if ids.shape[1] == 0:
        return "n/a (empty after tokenizing)", 0.0
    with torch.no_grad():
        logits = model(ids.to(device), lengths.to(device))
        probs = torch.softmax(logits, dim=1).squeeze(0)
    top_idx = probs.argmax().item()
    return idx_to_label[top_idx], probs[top_idx].item()

def compare_models(text: str):
    label_a, conf_a = predict_intent(model_a, text)
    label_b, conf_b = predict_intent(model_b, text)
    print(f'"{text}"')
    print(f"  Model A (from scratch):    {label_a:22s} (confidence {conf_a:.1%})")
    print(f"  Model B (pretrained):      {label_b:22s} (confidence {conf_b:.1%})")
    print()

SAMPLE_INPUTS = [
    "What's the total price with any discounts?",
    "Please take my number off your calling list.",
    "Yeah sure, sign me up, when can I start?",
    "Not right now, maybe later this year.",
    "Do you offer a certificate that's actually recognized?",
    "By the way, what's the weather like there?",
]

for s in SAMPLE_INPUTS:
    compare_models(s)

# Edit this and re-run the cell to try your own sentence:
compare_models("How much will it cost and when does it start?")

## 10. Key takeaways

**How an RNN works:**
- It processes a sequence one step at a time, carrying a **hidden state** forward as
  memory: `h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)`.
- The *same* weights are reused at every timestep — that's what "recurrent" means.
- Trained via **backpropagation through time**: unroll the recurrence, backprop through
  every step, accumulate gradients for the shared weights.
- Vanilla RNNs struggle with long sequences (vanishing/exploding gradients) — this is
  why **LSTM**, **GRU**, and eventually **Transformers** exist. The core idea of "carry
  state forward, update it at every step" is still there in LSTM/GRU; Transformers trade
  the sequential recurrence for parallel attention over the whole sequence instead.

**From scratch vs. pretrained:**
- "From scratch" = every weight, including what words mean, is learned only from your
  labeled data. Simple, no dependencies, but data-hungry.
- "Pretrained" = start from representations already learned on a much larger, general
  corpus (GloVe embeddings here; a pretrained LLM is the same idea at a much bigger
  scale). Usually faster to converge and better generalization with limited labeled
  data — the situation almost every real project is actually in.
- The right choice depends on how much labeled data you have and how well your domain's
  language matches what the pretrained resource was trained on.

**Where this fits in `AI_Telecalling`:** a lightweight intent classifier like this could
sit in front of the LLM call in `ai_telecaller_poc.ipynb` / `telephony_bridge/pipeline.py`
— e.g., instantly route `opt_out` to the suppression path (as `is_opt_out_request` already
does with keyword matching) or `fee_question`/`schedule_question`/`certification_question`
straight to the RAG lookup, without spending an LLM call deciding that. It's small, fast
(no GPU or API call needed at inference time), and — as shown above — can be trained on
a realistically small amount of labeled call data if you start from pretrained
embeddings.

### Exercises for further exploration
1. Swap `nn.RNN` for `nn.LSTM` or `nn.GRU` in `IntentRNN` (same constructor signature) —
   does either train more smoothly or reach higher test accuracy?
2. Set `freeze_embeddings=True` for Model B — does freezing the pretrained embeddings
   help or hurt on this dataset size? Why might that change with a *larger* dataset?
3. Try a larger GloVe dimension (`glove-wiki-gigaword-100` or `-300`) — does more
   pretrained information per word help here, or does the tiny dataset become the
   bottleneck regardless?
4. Add more labeled examples per class (or pull real anonymized transcripts from a test
   run of `telephony_bridge/`) and re-run both models — does Model A's disadvantage
   shrink as training data grows?
5. Add a confusion matrix over the test set for both models — which intents get
   confused with which, and why might that make sense given the wording?